In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/Brazilian E-Commerce Public'

In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/Brazilian E-Commerce Public/'

path


'/content/drive/MyDrive/Brazilian E-Commerce Public/'

In [ ]:
customer = pd.read_csv(path + 'olist_customers_dataset.csv')
geolocation = pd.read_csv(path +'olist_geolocation_dataset.csv')
order_items = pd.read_csv(path +'olist_order_items_dataset.csv')
order_payments = pd.read_csv(path +'olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(path +'olist_order_reviews_dataset.csv')
orders = pd.read_csv(path +'olist_orders_dataset.csv')
products = pd.read_csv(path +'olist_products_dataset.csv')
sellers = pd.read_csv(path +'olist_sellers_dataset.csv')
category = pd.read_csv(path +'product_category_name_translation.csv')

In [ ]:
# khảo sát các bảng
dfs = {
    'customer': customer,
    'geolocation': geolocation,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category': category
}

for name, df in dfs.items():
    print(f"\n{'='*50}")
    print(f"BẢNG: {name} | Shape: {df.shape}")
    print(f"{'='*50}")
    print(df.dtypes)
    print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nDuplicate rows: {df.duplicated().sum()}")


BẢNG: customer | Shape: (99441, 5)
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows: 0

BẢNG: geolocation | Shape: (1000163, 5)
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows: 261831

BẢNG: order_items | Shape: (112650, 7)
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows: 0

BẢNG: order_payments | Shape: (103886, 5)
order_id               

In [ ]:
# xử lý bảng geolocation
print(f"Số zip code unique: {geolocation['geolocation_zip_code_prefix'].nunique()}")
print(f"Tổng số dòng: {len(geolocation)}")

Số zip code unique: 19015
Tổng số dòng: 1000163


In [ ]:
geolocation_clean = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print(f"Sau khi gộp: {geolocation_clean.shape}")
print(geolocation_clean.head())

Sau khi gộp: (19015, 5)
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1001       -23.550190       -46.634024   
1                         1002       -23.548146       -46.634979   
2                         1003       -23.548994       -46.635731   
3                         1004       -23.549799       -46.634757   
4                         1005       -23.549456       -46.636733   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  


In [ ]:
print("Missing order_approved_at theo status:")
print(orders[orders['order_approved_at'].isnull()]['order_status'].value_counts())

print("\nMissing order_delivered_carrier_date theo status:")
print(orders[orders['order_delivered_carrier_date'].isnull()]['order_status'].value_counts())

print("\nMissing order_delivered_customer_date theo status:")
print(orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts())

Missing order_approved_at theo status:
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

Missing order_delivered_carrier_date theo status:
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

Missing order_delivered_customer_date theo status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [ ]:
delivered_missing = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isnull())
]
print(delivered_missing[['order_id', 'order_status', 'order_purchase_timestamp',
                          'order_approved_at', 'order_delivered_carrier_date',
                          'order_delivered_customer_date']])

                               order_id order_status order_purchase_timestamp  \
3002   2d1e2d5bf4dc7227b3bfebb81328c15f    delivered      2017-11-28 17:44:07   
20618  f5dd62b788049ad9fc0526e3ad11a097    delivered      2018-06-20 06:58:43   
43834  2ebdfc4f15f23b91474edf87475f108e    delivered      2018-07-01 17:05:11   
79263  e69f75a717d64fc5ecdfae42b2e8e086    delivered      2018-07-01 22:05:55   
82868  0d3268bad9b086af767785e3f0fc0133    delivered      2018-07-01 21:14:02   
92643  2d858f451373b04fb5c984a1cc2defaf    delivered      2017-05-25 23:22:43   
97647  ab7c89dc1bf4a1ead9d6ec1ec8968a84    delivered      2018-06-08 12:09:39   
98038  20edc82cf5400ce95e1afacc25798b31    delivered      2018-06-27 16:09:12   

         order_approved_at order_delivered_carrier_date  \
3002   2017-11-28 17:56:40          2017-11-30 18:12:23   
20618  2018-06-20 07:19:05          2018-06-25 08:05:00   
43834  2018-07-01 17:15:12          2018-07-03 13:57:00   
79263  2018-07-01 22:15:14        

In [ ]:
# Không điền giá trị giả (không dùng purchase_date, không dùng estimated_delivery_date để thay thế)
# vì sẽ làm sai lệch các phép tính "thời gian giao hàng thực tế"
# Giữ nguyên NaT - khi tính delivery time ở SQL, các dòng này sẽ tự động bị loại
# khi dùng điều kiện WHERE order_delivered_customer_date IS NOT NULL

# Chỉ cần ghi chú lại số liệu này để đưa vào README (thể hiện bạn đã kiểm tra data quality kỹ)
data_quality_notes = {
    'orders_delivered_missing_date': 8,
    'note': 'Status = delivered nhưng thiếu delivered_customer_date - giữ nguyên NaT, loại khỏi các phép tính delivery time'
}
print(data_quality_notes)

{'orders_delivered_missing_date': 8, 'note': 'Status = delivered nhưng thiếu delivered_customer_date - giữ nguyên NaT, loại khỏi các phép tính delivery time'}


In [ ]:
missing_category = products[products['product_category_name'].isnull()]
print(f"Số dòng thiếu category: {len(missing_category)}")
print("\nTrong số đó, số dòng cũng thiếu các cột khác:")
print(missing_category[['product_name_lenght', 'product_description_lenght', 'product_photos_qty']].isnull().sum())

Số dòng thiếu category: 610

Trong số đó, số dòng cũng thiếu các cột khác:
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64


In [ ]:
# Điền category thiếu = 'unknown' (không xóa dòng, vì product_id này vẫn có thể xuất hiện trong order_items)
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Merge với bảng dịch tên tiếng Anh
products = products.merge(category, on='product_category_name', how='left')

# Sau merge, các dòng 'unknown' sẽ không tìm được bản dịch -> tiếp tục điền 'unknown'
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

# Kiểm tra lại
print(products['product_category_name_english'].isnull().sum())  # phải = 0
print(products[products['product_category_name'] == 'unknown'].shape[0])  # phải = 610

0
610


In [ ]:
size_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

# Xem trước 2 dòng bị thiếu này là dòng nào
print(products[products[size_cols].isnull().any(axis=1)])

                             product_id product_category_name  \
8578   09ff539a621711667c43eba6a3bd8466                 bebes   
18851  5eb564652db742ff8f28759cd8d2652a               unknown   

       product_name_lenght  product_description_lenght  product_photos_qty  \
8578                  60.0                       865.0                 3.0   
18851                  NaN                         NaN                 NaN   

       product_weight_g  product_length_cm  product_height_cm  \
8578                NaN                NaN                NaN   
18851               NaN                NaN                NaN   

       product_width_cm product_category_name_english  
8578                NaN                          baby  
18851               NaN                       unknown  


In [ ]:
for col in size_cols:
    products[col] = products[col].fillna(products[col].median())

# Kiểm tra lại toàn bộ bảng products
print(products.isnull().sum())

product_id                         0
product_category_name              0
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   0
product_length_cm                  0
product_height_cm                  0
product_width_cm                   0
product_category_name_english      0
dtype: int64


In [ ]:
metadata_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']
for col in metadata_cols:
    products[col] = products[col].fillna(0)

# Kiểm tra lại toàn bộ
print(products.isnull().sum())
print(f"\nTổng số dòng: {products.shape[0]}, Số dòng còn null bất kỳ: {products.isnull().any(axis=1).sum()}")

product_id                       0
product_category_name            0
product_name_lenght              0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
dtype: int64

Tổng số dòng: 32951, Số dòng còn null bất kỳ: 0


In [ ]:
review_date_cols = ['review_creation_date', 'review_answer_timestamp']
for col in review_date_cols:
    order_reviews[col] = pd.to_datetime(order_reviews[col], errors='coerce')

print(order_reviews.dtypes)
print(f"\nSố dòng lỗi convert (nếu có): {order_reviews[review_date_cols].isnull().sum()}")

review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

Số dòng lỗi convert (nếu có): review_creation_date       0
review_answer_timestamp    0
dtype: int64


In [ ]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')
print(order_items.dtypes)

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object


In [ ]:
orphan_items = order_items[~order_items['order_id'].isin(orders['order_id'])]
orphan_orders = orders[~orders['customer_id'].isin(customer['customer_id'])]
orphan_payments = order_payments[~order_payments['order_id'].isin(orders['order_id'])]
orphan_reviews = order_reviews[~order_reviews['order_id'].isin(orders['order_id'])]
orphan_item_products = order_items[~order_items['product_id'].isin(products['product_id'])]
orphan_item_sellers = order_items[~order_items['seller_id'].isin(sellers['seller_id'])]

print(f"order_items thiếu order tương ứng: {len(orphan_items)}")
print(f"orders thiếu customer tương ứng: {len(orphan_orders)}")
print(f"order_payments thiếu order tương ứng: {len(orphan_payments)}")
print(f"order_reviews thiếu order tương ứng: {len(orphan_reviews)}")
print(f"order_items thiếu product tương ứng: {len(orphan_item_products)}")
print(f"order_items thiếu seller tương ứng: {len(orphan_item_sellers)}")

order_items thiếu order tương ứng: 0
orders thiếu customer tương ứng: 0
order_payments thiếu order tương ứng: 0
order_reviews thiếu order tương ứng: 0
order_items thiếu product tương ứng: 0
order_items thiếu seller tương ứng: 0


In [ ]:
customer.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_customers_dataset.csv', index=False)
geolocation_clean.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_geolocation_dataset.csv', index=False)
order_items.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_order_items_dataset.csv', index=False)
order_payments.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_order_payments_dataset.csv', index=False)
order_reviews.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_order_reviews_dataset.csv', index=False)
orders.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_orders_dataset.csv', index=False)
products.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_products_dataset.csv', index=False)
sellers.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_sellers_dataset.csv', index=False)

print("Đã lưu xong tất cả các bảng vào thư mục!")

Đã lưu xong tất cả các bảng vào thư mục!


In [ ]:
geolocation_clean = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_state': 'first'
}).reset_index()

print(f"Sau khi gộp: {geolocation_clean.shape}")
print(geolocation_clean.head())

Sau khi gộp: (19015, 4)
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1001       -23.550190       -46.634024   
1                         1002       -23.548146       -46.634979   
2                         1003       -23.548994       -46.635731   
3                         1004       -23.549799       -46.634757   
4                         1005       -23.549456       -46.636733   

  geolocation_state  
0                SP  
1                SP  
2                SP  
3                SP  
4                SP  


In [ ]:
geolocation_clean.to_csv('D:\\Project\\Brazilian E-Commerce Public clean\\olist_geolocation_dataset.csv', index=False)